# Wikipedia Scraping for MEP Data

This notebook provides one primary, rerunnable workflow to scrape native-language and English Wikipedia pages for MEPs, plus archived legacy variants kept for reference.

## How to use this notebook

1. Run cells from top to bottom.
2. Update only the path setup variables (`USERNAME`, `PROJECT_FOLDER_NAME`, and optional file names) if your folder structure changes.
3. The main outputs are saved in the project `outputs/tables` folder.

The archived code at the end is intentionally non-executable and preserved for traceability.

What this section does:
- Clarifies the purpose of this section and how it connects to the next steps.
- Summarizes expected inputs and outputs to make reruns easier to follow.


In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Optional, Tuple
import csv
import difflib
import logging
import re
import sys
import time
import unicodedata
from urllib.parse import quote

import pandas as pd
import requests

# Ensure project root is importable in notebook kernels.
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src").exists() and (candidate / "README.md").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from src.config import PROJECT_ROOT, ensure_default_directories

ensure_default_directories()


## 1) Environment setup

Installs the required Python packages for this notebook (`pandas`, `openpyxl`, `requests`).
If packages are already installed, this cell can be skipped.

What this section does:
- Clarifies the purpose of this section and how it connects to the next steps.
- Summarizes expected inputs and outputs to make reruns easier to follow.


In [ ]:
from src.config import PROJECT_ROOT

WIKIPEDIA_DIR = PROJECT_ROOT / "3 Wikipedia"
MEP_LIST_DIR = PROJECT_ROOT / "2 MEP List"
DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = WIKIPEDIA_DIR / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
TABLES_DIR = OUTPUTS_DIR / "tables"

for folder in [DATA_DIR, OUTPUTS_DIR, FIGURES_DIR, TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

INPUT_CANDIDATES = [
    MEP_LIST_DIR / "MEP_2009_2019.xls",
    MEP_LIST_DIR / "1729084703_MEP_list Oct 2024.xlsx",
    WIKIPEDIA_DIR / "mep_wikipedia_pages_2.xlsx",
]
EXCEL_PATH = next((p for p in INPUT_CANDIDATES if p.exists()), INPUT_CANDIDATES[0])

OUTPUT_CSV = TABLES_DIR / "mep_wikipedia_pages.csv"
OUTPUT_XLSX = TABLES_DIR / "mep_wikipedia_pages.xlsx"

assert PROJECT_ROOT.exists(), f"Project root does not exist: {PROJECT_ROOT}"
assert EXCEL_PATH.exists(), (
    "No input Excel file found. Expected one of: "
    + ", ".join(str(p) for p in INPUT_CANDIDATES)
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Input Excel:  {EXCEL_PATH}")
print(f"CSV output:   {OUTPUT_CSV}")
print(f"XLSX output:  {OUTPUT_XLSX}")

## 2) Main scraper logic

The next cell contains the primary workflow used for final outputs.
It keeps the original analytical logic (fuzzy title resolution, native-language + English retrieval, incremental behavior), but now uses the shared path setup and cleaner configuration.

What this section does:
- Clarifies the purpose of this section and how it connects to the next steps.
- Summarizes expected inputs and outputs to make reruns easier to follow.


In [ ]:

# -------- Configuration --------
USER_AGENT = "MEP-Wiki-Scraper/1.7 (contact: you@example.com)"
REQUEST_PAUSE_SECONDS = 0.5
HEADERS = {"User-Agent": USER_AGENT}
SEARCH_LIMIT = 10

# Uses EXCEL_PATH, OUTPUT_CSV, OUTPUT_XLSX from the path setup cell.
EXCEL_PATH = Path(EXCEL_PATH)
OUTPUT_CSV = Path(OUTPUT_CSV)
OUTPUT_XLSX = Path(OUTPUT_XLSX)

# -------- Country -> Wikipedia language --------
COUNTRY_TO_LANG = {
    "Finland": "fi",
    "Spain": "es",
    "Poland": "pl",
    "Greece": "el",
    "Malta": "mt",
    "France": "fr",
    "Germany": "de",
    "Italy": "it",
    "Portugal": "pt",
    "Netherlands": "nl",
    "Belgium": "fr",
    "Sweden": "sv",
    "Denmark": "da",
    "Czechia": "cs",
    "Czech Republic": "cs",
    "Slovakia": "sk",
    "Slovenia": "sl",
    "Austria": "de",
    "Ireland": "ga",
    "Romania": "ro",
    "Bulgaria": "bg",
    "Hungary": "hu",
    "Croatia": "hr",
    "Lithuania": "lt",
    "Latvia": "lv",
    "Estonia": "et",
    "Cyprus": "el",
    "Luxembourg": "lb",
}

# -------- Load MEPs from Excel --------
def load_meps_from_excel(path: Path, n: int = 999999) -> List[dict]:
    if not path.exists():
        raise FileNotFoundError(f"Excel file not found at: {path}")

    df = pd.read_excel(path, dtype=str)

    # Supports either naming convention found in previous notebook versions.
    if {"Name", "Country"}.issubset(df.columns):
        name_col, country_col = "Name", "Country"
    elif {"name", "Country"}.issubset(df.columns):
        name_col, country_col = "name", "Country"
    elif {"full_name", "country"}.issubset(df.columns):
        name_col, country_col = "full_name", "country"
    else:
        raise KeyError(
            "Missing expected name/country columns. "
            f"Present columns: {list(df.columns)}"
        )

    df = df[[name_col, country_col]].head(n).copy()
    df[name_col] = df[name_col].fillna("").astype(str).str.strip()
    df[country_col] = df[country_col].fillna("").astype(str).str.strip()
    df = df[(df[name_col] != "") & (df[country_col] != "")]

    return [
        {"fullName": row[0], "country": row[1]}
        for row in df[[name_col, country_col]].itertuples(index=False, name=None)
    ]

# -------- Helpers --------
def strip_diacritics(s: str) -> str:
    return "".join(ch for ch in unicodedata.normalize("NFD", s) if unicodedata.category(ch) != "Mn")


def simplify(s: str) -> str:
    s = strip_diacritics(s.lower())
    s = re.sub(r"[^\w\s]", " ", s, flags=re.UNICODE)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def normalise_name_case(name: str) -> str:
    parts = name.split()
    return " ".join(p.capitalize() if p.isupper() else p for p in parts)


def generate_name_variants(name: str) -> List[str]:
    variants: List[str] = []
    base = name.strip()

    def add(v: str) -> None:
        if v and v not in variants:
            variants.append(v)

    add(base)
    add(normalise_name_case(base))
    no_diac = strip_diacritics(base)
    add(no_diac)
    add(normalise_name_case(no_diac))
    add(" ".join(base.split()))

    tokens = base.split()
    if len(tokens) >= 2:
        first, last = tokens[0], tokens[-1]
        add(f"{first} {last}")
        add(f"{last} {first}")
        add(strip_diacritics(f"{first} {last}"))
        add(strip_diacritics(f"{last} {first}"))
        add(normalise_name_case(f"{first} {last}"))
        add(normalise_name_case(f"{last} {first}"))

    replacements = [
        ("Georgios", "Giorgos"),
        ("Giorgos", "Georgios"),
        ("Georgios", "Yorgos"),
        ("Yorgos", "Georgios"),
    ]
    for old, new in replacements:
        if old in base:
            add(base.replace(old, new))
            add(normalise_name_case(base.replace(old, new)))

    return variants


def mw_query(lang: str, params: Dict) -> Dict:
    base = f"https://{lang}.wikipedia.org/w/api.php"
    response = requests.get(base, params=params, headers=HEADERS, timeout=30)
    response.raise_for_status()
    return response.json()


def mw_exact_title_if_exists(lang: str, title: str) -> Optional[str]:
    data = mw_query(
        lang,
        {
            "action": "query",
            "format": "json",
            "prop": "info",
            "redirects": "1",
            "titles": title,
            "inprop": "url",
        },
    )
    pages = data.get("query", {}).get("pages", {})
    for pid, page in pages.items():
        if str(pid).lstrip("-").isdigit() and int(pid) > 0 and "missing" not in page:
            return page.get("title")
    return None


def mw_search_candidates(lang: str, query: str, limit: int = SEARCH_LIMIT) -> List[str]:
    titles: List[str] = []

    data = mw_query(
        lang,
        {
            "action": "query",
            "format": "json",
            "list": "search",
            "srsearch": query,
            "srlimit": str(limit),
            "utf8": "1",
            "srprop": "",
        },
    )
    for hit in data.get("query", {}).get("search", []):
        title = hit.get("title")
        if title and title not in titles:
            titles.append(title)

    data2 = mw_query(
        lang,
        {
            "action": "query",
            "format": "json",
            "list": "prefixsearch",
            "pssearch": query,
            "pslimit": str(limit),
        },
    )
    for hit in data2.get("query", {}).get("prefixsearch", []):
        title = hit.get("title")
        if title and title not in titles:
            titles.append(title)

    return titles


def score_title(candidate: str, target_name: str) -> float:
    a = simplify(candidate)
    b = simplify(target_name)
    score = difflib.SequenceMatcher(None, a, b).ratio()
    if a.startswith(b) or b.startswith(a):
        score += 0.15
    ta, tb = set(a.split()), set(b.split())
    if ta and tb:
        overlap = len(ta & tb) / max(len(ta), 1)
        score += 0.2 * overlap
    return min(score, 1.0)


def resolve_title_fuzzy(lang: str, name: str) -> Optional[str]:
    for candidate in generate_name_variants(name):
        exact = mw_exact_title_if_exists(lang, candidate)
        if exact:
            return exact

    pool: List[str] = []
    queries: List[str] = []
    for candidate in generate_name_variants(name):
        queries.extend([f'"{candidate}"', f'intitle:"{candidate}"', candidate])

    for query in queries:
        for title in mw_search_candidates(lang, query, limit=SEARCH_LIMIT):
            if title not in pool:
                pool.append(title)
        if len(pool) >= SEARCH_LIMIT:
            break

    if not pool:
        return None

    best = max(pool, key=lambda t: score_title(t, name))
    return best if score_title(best, name) >= 0.62 else None


def fetch_html_and_wikitext(lang: str, title: str) -> Tuple[str, str, str, Dict[str, dict]]:
    data = mw_query(
        lang,
        {
            "action": "parse",
            "page": title,
            "prop": "text|displaytitle|langlinks",
            "format": "json",
            "formatversion": "2",
            "redirects": "1",
            "lllimit": "500",
        },
    )
    if "parse" not in data:
        return "", "", "", {}

    html = data["parse"].get("text") or ""
    display_title = data["parse"].get("title") or title
    url = f"https://{lang}.wikipedia.org/wiki/{quote(display_title.replace(' ', '_'))}"

    langlinks_raw = data["parse"].get("langlinks") or []
    langlinks: Dict[str, dict] = {}
    for ll in langlinks_raw:
        code = ll.get("lang")
        if not code:
            continue
        ll_title = ll.get("title") or ll.get("*") or ""
        ll_url = ll.get("url") or (
            f"https://{code}.wikipedia.org/wiki/{quote(ll_title.replace(' ', '_'))}" if ll_title else ""
        )
        langlinks[code] = {"title": ll_title, "url": ll_url}

    data_w = mw_query(
        lang,
        {
            "action": "parse",
            "page": title,
            "prop": "wikitext",
            "format": "json",
            "formatversion": "2",
            "redirects": "1",
        },
    )
    wikitext_val = ""
    if "parse" in data_w:
        wt = data_w["parse"].get("wikitext")
        if isinstance(wt, str):
            wikitext_val = wt
        elif isinstance(wt, dict):
            wikitext_val = wt.get("*", "") or wt.get("content", "")

    return html, wikitext_val, url, langlinks


def resolve_via_english_bridge(native_lang: str, name: str) -> Optional[str]:
    en_title = resolve_title_fuzzy("en", name)
    if not en_title:
        return None
    _, _, _, langlinks = fetch_html_and_wikitext("en", en_title)
    native = langlinks.get(native_lang)
    return native["title"] if native and native.get("title") else None


def fetch_native_side(lang: Optional[str], name: str) -> Tuple[str, str, str, str, Dict[str, dict]]:
    if not lang:
        return "", "", "", "", {}
    title_native = resolve_title_fuzzy(lang, name)
    if not title_native:
        bridged = resolve_via_english_bridge(lang, name)
        if bridged:
            title_native = bridged
    if not title_native:
        return "", "", "", "", {}
    html_native, wiki_native, url_native, langlinks = fetch_html_and_wikitext(lang, title_native)
    return title_native or "", url_native or "", html_native or "", wiki_native or "", langlinks


def fetch_english_side(
    name: str,
    langlinks_from_native: Optional[Dict[str, dict]] = None,
    ) -> Tuple[str, str, str, str]:
    page_title_en = ""
    page_url_en = ""

    if langlinks_from_native and "en" in langlinks_from_native and langlinks_from_native["en"].get("title"):
        page_title_en = langlinks_from_native["en"]["title"] or ""
        page_url_en = langlinks_from_native["en"].get("url") or ""

    if not page_title_en:
        maybe_en = resolve_title_fuzzy("en", name)
        if maybe_en:
            page_title_en = maybe_en

    html_en, wiki_en = "", ""
    if page_title_en:
        html_en, wiki_en, url_en, _ = fetch_html_and_wikitext("en", page_title_en)
        if url_en:
            page_url_en = url_en

    return page_title_en or "", page_url_en or "", html_en or "", wiki_en or ""


def main() -> None:
    logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

    meps_from_file = load_meps_from_excel(EXCEL_PATH, n=999999)
    if not meps_from_file:
        raise RuntimeError("No MEP rows loaded from Excel. Check name/country columns in the input file.")

    if OUTPUT_XLSX.exists():
        existing_df = pd.read_excel(OUTPUT_XLSX, dtype=str)
        logging.info("Loaded existing output: %s (%d rows)", OUTPUT_XLSX, len(existing_df))
    elif OUTPUT_CSV.exists():
        existing_df = pd.read_csv(OUTPUT_CSV, dtype=str)
        logging.info("Loaded existing output: %s (%d rows)", OUTPUT_CSV, len(existing_df))
    else:
        existing_df = pd.DataFrame(
            columns=[
                "name",
                "wikipedia_language",
                "page_title",
                "page_url",
                "wikipedia_content_html",
                "wikipedia_content_wikitext",
                "page_title_en",
                "page_url_en",
                "wikipedia_content_html_en",
                "wikipedia_content_wikitext_en",
            ]
        )
        logging.info("No existing output found, starting fresh.")

    if "name" not in existing_df.columns:
        existing_df["name"] = ""

    existing_names = set(existing_df["name"].dropna().astype(str).str.strip())
    rows = existing_df.to_dict(orient="records")

    for mep in meps_from_file:
        name = mep["fullName"].strip()
        country = mep["country"].strip()
        lang = COUNTRY_TO_LANG.get(country)

        if name in existing_names:
            logging.info("Skipping %s (already in output)", name)
            continue

        logging.info("Processing %s [%s -> %s + en]", name, country, lang or "N/A")

        page_title_native, page_url_native, html_native, wiki_native, langlinks_native = fetch_native_side(lang, name)
        page_title_en, page_url_en, html_en, wiki_en = fetch_english_side(
            name,
            langlinks_from_native=langlinks_native if langlinks_native else None,
        )

        rows.append(
            {
                "name": name,
                "wikipedia_language": lang or "",
                "page_title": page_title_native,
                "page_url": page_url_native,
                "wikipedia_content_html": html_native,
                "wikipedia_content_wikitext": wiki_native,
                "page_title_en": page_title_en,
                "page_url_en": page_url_en,
                "wikipedia_content_html_en": html_en,
                "wikipedia_content_wikitext_en": wiki_en,
            }
        )
        existing_names.add(name)
        time.sleep(REQUEST_PAUSE_SECONDS)

    out_df = pd.DataFrame(
        rows,
        columns=[
            "name",
            "wikipedia_language",
            "page_title",
            "page_url",
            "wikipedia_content_html",
            "wikipedia_content_wikitext",
            "page_title_en",
            "page_url_en",
            "wikipedia_content_html_en",
            "wikipedia_content_wikitext_en",
        ],
    )

    out_df = out_df.drop_duplicates(subset=["name"], keep="first").reset_index(drop=True)
    out_df.to_csv(OUTPUT_CSV, index=False, quoting=csv.QUOTE_ALL, encoding="utf-8-sig")
    out_df.to_excel(OUTPUT_XLSX, index=False)

    print(f"Saved {len(out_df)} rows")
    print(f"CSV:  {OUTPUT_CSV}")
    print(f"XLSX: {OUTPUT_XLSX}")


main()


INFO: Loaded existing output: C:\Users\Gebruiker\OneDrive - UniversitÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â  Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\MEP\4 Wikipedia\mep_wikipedia_pages.xlsx (1916 rows)
INFO: Skipping BARBAGALLO Giovanni (already in output)
INFO: Skipping CRUZ Joaquim (already in output)
INFO: Skipping DIETRICH Roberto (already in output)
INFO: Skipping GKIKOPOULOS Georgios (already in output)
INFO: Skipping HIDVÃƒÆ’Ã†â€™ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â°GHI BalÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¡zs (already in output)
INFO: Skipping KARI Rina Ronja (already in output)
INFO: Skipping KOEWIUS Annette (already in output)
INFO: Skipping REVAULT D'ALLONNES BONNEFOY Christine (already in output)
INFO: Skipping ABELA BALDACCHINO Claudette (already in output)
INFO: Skipping BALDINI Marino (already in output)


Saved 1916 rows to C:\Users\Gebruiker\OneDrive - UniversitÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â  Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\MEP\4 Wikipedia\mep_wikipedia_pages.csv and C:\Users\Gebruiker\OneDrive - UniversitÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â  Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\MEP\4 Wikipedia\mep_wikipedia_pages.xlsx


In [ ]:
assert OUTPUT_CSV.parent.exists(), f"Missing output folder: {OUTPUT_CSV.parent}"
assert OUTPUT_XLSX.parent.exists(), f"Missing output folder: {OUTPUT_XLSX.parent}"

if "df_out" in globals():
    assert len(df_out) > 0, "Scraping output dataframe is empty"
    if {"name", "wikipedia_language"}.issubset(df_out.columns):
        dup = int(df_out.duplicated(["name", "wikipedia_language"]).sum())
        print(f"Duplicate rows on name-language keys: {dup}")

if OUTPUT_CSV.exists() and OUTPUT_XLSX.exists():
    print("Output exports exist and are ready for downstream notebooks.")

## 3) Quick output checks

Run a lightweight validation on the saved table to confirm row count, column presence, and missing values in key fields before hand-in.

What this section does:
- Clarifies the purpose of this section and how it connects to the next steps.
- Summarizes expected inputs and outputs to make reruns easier to follow.


In [ ]:

check_df = pd.read_excel(OUTPUT_XLSX, dtype=str) if OUTPUT_XLSX.exists() else pd.DataFrame()

print(f"Rows: {len(check_df)}")
print(f"Columns: {len(check_df.columns)}")

if not check_df.empty:
    key_cols = ["name", "wikipedia_language", "page_url", "page_url_en"]
    existing_key_cols = [c for c in key_cols if c in check_df.columns]
    missing_summary = check_df[existing_key_cols].isna().sum().sort_values(ascending=False)
    print("Missing values in key columns:")
    display(missing_summary.to_frame("missing_count"))

print(f"Saved tables folder: {TABLES_DIR}")
